# Excel custom parser walkthrough

This notebook shows how to parse existing MARIO-readable Excel workbooks with `mario.parse_from_excel(...)`. It uses the small test workbooks shipped with MARIO, so the focus is on reading a completed workbook rather than generating a new template.

## What this notebook covers

- parse standard IOT and SUT Excel workbooks;
- parse Excel workbooks that need explicit `matrix_layouts`;
- inspect the parsed baseline blocks and index labels.

Creating a blank workbook from sets and units is covered in the dedicated workflow on providing a custom database.

## Example workbooks

The examples use these packaged workbooks:

- `test_IOT_standard.xlsx`: standard IOT layout;
- `test_IOT_special.xlsx`: IOT layout with extra semantic levels on `V` and `E`;
- `test_SUT_standard.xlsx`: standard SUT layout;
- `test_SUT_special.xlsx`: SUT layout with extra semantic levels on satellite extensions.

The code below uses paths relative to this notebook source location in the MARIO repository. If you run the notebook elsewhere, replace the path strings with the location of your local Excel files.

In [1]:
import mario

## Parse a standard IOT workbook

For a standard IOT workbook, only the file path, table kind and mode are required.

In [2]:
iot = mario.parse_from_excel(
    path="../test_IOT_standard.xlsx",
    table="IOT",
    mode="flows",
)

iot.meta.table, iot.get_index("Region"), iot.get_index("Sector"), sorted(iot["baseline"].keys())

INFO Parser: excel reading IOT flows from /path/to/test_IOT_standard.xlsx.


INFO Parser: state payload ready with 6 canonical blocks.


INFO Parser: excel state ready for IOT.


INFO Metadata: initialized.


('IOT',
 ['Reg1', 'Reg2'],
 ['Agriculture', 'Services', 'Industry'],
 ['E', 'EY', 'V', 'VY', 'Y', 'Z'])

## Parse an IOT workbook with custom matrix layouts

Use `matrix_layouts` when specific matrices carry extra semantic levels beyond the default layout. In this example, `V` is indexed by region and sector, while `E` is indexed by region.

In [3]:
iot_special = mario.parse_from_excel(
    path="../test_IOT_special.xlsx",
    table="IOT",
    mode="flows",
    matrix_layouts={
        "V": ("Region", "Sector"),
        "E": "Region",
    },
    calc_all=False,
)

iot_special["baseline"]["V"].index.names, iot_special["baseline"]["E"].index.names

INFO Parser: excel reading IOT flows from /path/to/test_IOT_special.xlsx.


INFO Parser: state payload ready with 6 canonical blocks.


INFO Parser: excel state ready for IOT.


INFO Metadata: initialized.


(FrozenList(['Region', 'Sector', 'Factor of production']),
 FrozenList(['Region', 'Satellite account']))

## Parse a standard SUT workbook

For SUT workbooks, pass the table kind as `"SUT"`. You can also pass `tech_assumption=` when you want MARIO to record the technology assumption used downstream.

In [4]:
sut = mario.parse_from_excel(
    path="../test_SUT_standard.xlsx",
    table="SUT",
    mode="flows",
    tech_assumption="IT",
    calc_all=False,
)

sut.meta.table, sorted(sut["baseline"].keys()), sut["baseline"]["S"].shape, sut["baseline"]["U"].shape

INFO Parser: excel reading SUT flows from /path/to/test_SUT_standard.xlsx.


INFO Parser: state payload ready with 10 canonical blocks.


INFO Parser: excel state ready for SUT.


INFO Metadata: initialized.


('SUT',
 ['EY', 'Ea', 'Ec', 'S', 'U', 'VY', 'Va', 'Vc', 'Ya', 'Yc'],
 (4, 4),
 (4, 4))

## Parse a SUT workbook with custom matrix layouts

The same `matrix_layouts` mechanism applies to SUT workbooks. Here, the satellite-account matrices are declared with region and commodity detail.

In [5]:
sut_special = mario.parse_from_excel(
    path="../test_SUT_special.xlsx",
    table="SUT",
    mode="flows",
    tech_assumption="IT",
    matrix_layouts={
        "E": ("Region", "Commodity"),
    },
    calc_all=False,
)

sut_special["baseline"]["Ec"].index.names, sut_special["baseline"]["Ea"].index.names

INFO Parser: excel reading SUT flows from /path/to/test_SUT_special.xlsx.


INFO Parser: state payload ready with 10 canonical blocks.


INFO Parser: excel state ready for SUT.


INFO Metadata: initialized.


(FrozenList(['Region', 'Commodity', 'Satellite account']),
 FrozenList(['Region', 'Commodity', 'Satellite account']))

## Practical notes

- `mode="flows"` reads flow matrices; use `mode="coefficients"` only for coefficient workbooks.
- `matrix_layouts` must use MARIO semantic index names, such as `Region`, `Sector`, `Activity`, `Commodity`, `Factor of production`, `Final demand`, and `Satellite account`.
- Fully empty matrices are accepted and filled with zeros; partially empty matrices are treated as invalid input.